In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')
DRIVE_BASE = '/content/drive/MyDrive/LoRA_Compress'
os.makedirs(DRIVE_BASE, exist_ok=True)
os.makedirs(f'{DRIVE_BASE}/databases', exist_ok=True)
os.makedirs(f'{DRIVE_BASE}/results', exist_ok=True)

In [ ]:
%cd /content
!git clone https://github.com/qades/loracompress.git 2>/dev/null || true
%cd loracompress
!pip install -q transformers torch optuna tqdm bitsandbytes accelerate

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import optuna
import random
import numpy as np
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
print(f'PyTorch: {torch.__version__}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')

In [ ]:
MODEL_NAME = 'HuggingFaceTB/SmolLM2-135M'
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
)
print('Loading Q4 model...')
model_q4 = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=quantization_config,
    device_map='auto', trust_remote_code=True)
print('Q4 model loaded')

In [ ]:
print('Loading FP16 model...')
model_fp16 = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.float32, device_map='cpu',
    trust_remote_code=True)
print('FP16 model loaded')

In [ ]:
ANALYSIS_RANKS = [16, 32, 64, 128]
PRIMARY_RANK = 32

def get_svd_stats(weight, ranks):
    w = weight.float()
    try:
        if torch.all(w == 0) or torch.std(w) < 1e-10:
            return {r: {'var': 0.0, 'l1_err': 100.0} for r in ranks}
        U, S, Vh = torch.linalg.svd(w, full_matrices=False)
        total = (S ** 2).sum()
        if total < 1e-10:
            return {r: {'var': 0.0, 'l1_err': 100.0} for r in ranks}
        results = {}
        for r in ranks:
            if r <= len(S):
                var = (S[:r] ** 2).sum() / total * 100
                S_r = torch.zeros_like(S)
                S_r[:r] = S[:r]
                W_r = U @ torch.diag(S_r) @ Vh
                l1_err = torch.mean(torch.abs(W_r - w)).item() / torch.mean(torch.abs(w)).item() * 100
                results[r] = {'var': min(100.0, max(0.0, var)), 'l1_err': min(1000.0, max(0.0, l1_err))}
            else:
                results[r] = {'var': 100.0, 'l1_err': 0.0}
        return results
    except:
        return {r: {'var': 0.0, 'l1_err': 100.0} for r in ranks}

def get_q4_weight(param):
    try:
        w = param.data.cpu().float()
        return w
    except:
        return None

print('Analyzing Q4 vs FP16...')
layer_comparisons = []

for (name_q4, p_q4), (name_fp16, p_fp16) in zip(model_q4.named_parameters(), model_fp16.named_parameters()):
    if len(p_q4.shape) != 2 or p_q4.shape[0] < 100:
        continue
    w_q4 = get_q4_weight(p_q4)
    if w_q4 is None:
        continue
    w_fp16 = p_fp16.data.cpu().float()
    if w_q4.shape != w_fp16.shape:
        continue
    stats_q4 = get_svd_stats(w_q4, ANALYSIS_RANKS)
    stats_fp16 = get_svd_stats(w_fp16, ANALYSIS_RANKS)
    q4_vars = [stats_q4[r]['var'] for r in ANALYSIS_RANKS]
    if all(v > 99 for v in q4_vars):
        print(f'Warning: {name_q4} has suspicious 100% variance')
    q4_easier = stats_q4[PRIMARY_RANK]['var'] > stats_fp16[PRIMARY_RANK]['var']
    advantage = stats_q4[PRIMARY_RANK]['var'] - stats_fp16[PRIMARY_RANK]['var']
    layer_comparisons.append({
        'name': name_q4, 'shape': tuple(p_q4.shape),
        'stats_q4': stats_q4, 'stats_fp16': stats_fp16,
        'q4_easier': q4_easier, 'advantage': advantage,
        'w_q4': w_q4, 'w_fp16': w_fp16})

print(f'Analyzed {len(layer_comparisons)} layers')
q4_easier_count = sum(1 for l in layer_comparisons if l['q4_easier'])
print(f'Q4 has advantage in {q4_easier_count}/{len(layer_comparisons)} layers')

In [ ]:
if layer_comparisons:
    test = layer_comparisons[0]
    print(f'Layer: {test["name"]}')
    print(f'Q4 shape: {test["w_q4"].shape}, dtype: {test["w_q4"].dtype}')
    print(f'Q4 stats: mean={test["w_q4"].mean():.6f}, std={test["w_q4"].std():.6f}')
    print(f'Unique values: {len(torch.unique(test["w_q4"]))}')
    print(f'FP16 stats: mean={test["w_fp16"].mean():.6f}, std={test["w_fp16"].std():.6f}')
    if test['w_q4'].std() < 0.01:
        print('WARNING: Q4 weights have very low variance!')
    else:
        print('Q4 weights look normal')